# Data Profiling and Quality Checks
Goal: Check raw freMTPL2 data quality and build a clean policy-level severity table for EDA.

- I. Imports & config (paths, seed)
- II. Data loading (freq + sev)
- III. Schema & basic info (shape, dtypes, head, fix types)
- IV. Quality checks
   - IV.a Duplicates
   - IV.b Missing values
   - IV.c Value ranges & cardinality
   - IV.d Join integrity (freq ↔ sev)
   - IV.e Outlier flagging
- V. Build policy-level severity table (aggregate, join, target, weight)
- VI. Findings & decisions
- VII. Save to data/processed/

# I. Imports and config

In [ ]:
# Import libraries
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# SETUP AND CONFIGURATION
from pathlib import Path


# Locate project root via a marker file
def find_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from the current directory until the marker file is found."""
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


# Define project root and data directories

# Project root directory
PROJECT_ROOT = find_project_root()

# Data directories
DATA_RAW = PROJECT_ROOT / "data" / "raw"  # Raw data directory

# II. Data Loading

In [ ]:
# Import freMTPL2 data from sklearn
from sklearn.datasets import fetch_openml

# Start timer
start_time = time.time()

# freMTPL2 from OpenML: policies (frequency) and individual claims (severity)
freq = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame
sev = fetch_openml(data_id=41215, as_frame=True, parser="auto").frame

# End timer and print elapsed time
end_time = time.time()
print(f"Data fetched in {end_time - start_time:.2f} seconds")

# III. Schema & basic info (shape, dtypes, head, fix types)

In [ ]:
# Shape of the datasets
print(f"Frequency dataset shape: {freq.shape}")
print(f"Severity dataset shape: {sev.shape}")

In [ ]:
# Display the first few rows of the frequency dataset
print("Frequency dataset preview:")
freq.head()

In [ ]:
# Display the first few rows of the severity dataset
print("Severity dataset preview:")
sev.head()

In [ ]:
# Display information about the frequency dataset
print("Frequency dataset information:")
freq.info()

In [ ]:
# Display information about the severity dataset
print("Severity dataset information:")
sev.info()

In [ ]:
# Fix types
# Convert 'IDpol' column to integer type in the frequency dataset
freq["IDpol"] = freq["IDpol"].astype(int)

# Strip the literal quote characters from VehGas and convert to category
freq["VehGas"] = freq["VehGas"].str.strip("'").astype("category")

# Check dtypes after conversion
print("Frequency dataset dtypes after conversion:")
print(freq.dtypes)

print("\nSeverity dataset dtypes after conversion:")
print(sev.dtypes)

# IV. Quality checks

### IV.a Duplicates

In [ ]:
# Check for full-row duplicates in the frequency dataset
freq_duplicates = freq.duplicated(keep=False).sum()
print(f"Number of duplicate rows in the frequency dataset: {freq_duplicates}")

# Check for duplicates in IDpol column in the frequency dataset
idpol_duplicates = freq["IDpol"].duplicated(keep=False).sum()
print(f"Number of duplicate IDpol values in the frequency dataset: {idpol_duplicates}")

# Check for full-row duplicates in the severity dataset
sev_duplicates = sev.duplicated(keep=False).sum()
print(f"\nNumber of duplicate rows in the severity dataset: {sev_duplicates}")

# Check for duplicates in IDpol column in the severity dataset
idpol_sev_duplicates = sev["IDpol"].duplicated(keep=False).sum()
print(f"Number of duplicate IDpol values in the severity dataset: {idpol_sev_duplicates}")

In [ ]:
# Investigate duplicates in the severity dataset
# Display the duplicate rows in the severity dataset
if sev_duplicates > 0:
    print("Duplicate rows in the severity dataset:")
    print(sev[sev.duplicated(keep=False)])

In [ ]:
# Group the severity dataset by 'IDpol' and
# count the number of claims per policy along with the total claim amount
sev_grouped = (
    sev.groupby("IDpol")
    .agg(n_claims=("ClaimAmount", "count"), total_claim_amount=("ClaimAmount", "sum"))
    .reset_index()
)

# Display the grouped severity dataset
print("Grouped severity dataset (by IDpol):")
sev_grouped.head()

In [ ]:
# Check for full-row duplicates in the grouped severity dataset
grouped_duplicates = sev_grouped.duplicated(keep=False).sum()
print(f"Number of duplicate rows in the grouped severity dataset: {grouped_duplicates}")

# Check for duplicates in IDpol column in the grouped severity dataset
idpol_grouped_duplicates = sev_grouped["IDpol"].duplicated(keep=False).sum()
print(f"Number of duplicate IDpol values in the grouped severity data: {idpol_grouped_duplicates}")

### IV.b Missing values

In [ ]:
# Check for missing values in both datasets
# Check for missing values in the frequency dataset
freq_missing = freq.isnull().sum()
print("Missing values in the frequency dataset:")
print(freq_missing)

# Check for missing values in the severity dataset
sev_missing = sev.isnull().sum()
print("\nMissing values in the severity dataset:")
print(sev_missing)

### IV.c Value ranges & cardinality

### IV.c Value ranges & cardinality

Known issue: ✅ none · ⚠️ breaks the rule · 🔍 passes the rule but worth a look

**Frequency table (`freq`): one row per policy**

| Feature | Type | Meaning | Allowed min | Allowed max | Check | Known issue |
|---|---|---|---|---|---|---|
| `IDpol` | ID | Policy ID, unique per row | n/a | n/a | Must be unique | ✅ |
| `ClaimNb` | Count | Number of claims in the period | 0 | No limit | Cannot be negative | ✅ |
| `Exposure` | Numeric | Fraction of a year the policy was in force | > 0 | 1 | Must be above 0 and at most 1 | ⚠️ 1,224 rows above 1 |
| `Area` | Categorical (ordered) | Density band of the driver's area | A | F | Must be one of A to F | ✅ |
| `VehPower` | Numeric (ordered) | Power level of the car | > 0 | No limit | Must be positive | ✅ |
| `VehAge` | Numeric | Vehicle age in years | 0 | No limit | Cannot be negative | 🔍 1,116 rows above 30 years |
| `DrivAge` | Numeric | Driver age in years | 18 | No limit | Must be at least 18 (French licensing age when the data was collected) | ✅ |
| `BonusMalus` | Numeric | No-claims discount / penalty, 100 = base | 50 | 350 | Must be between 50 and 350 | ✅ |
| `VehBrand` | Categorical | Anonymized brand group | n/a | n/a | Must be a known brand code | ✅ |
| `VehGas` | Categorical | Fuel type (Regular = petrol) | n/a | n/a | Must be Diesel or Regular | ✅ |
| `Density` | Numeric | Inhabitants per km² in the driver's town | > 0 | No limit | Must be positive | ✅ |
| `Region` | Categorical | French region code | n/a | n/a | Must be a known region code | ✅ |

**Severity table (`sev`): one row per claim**

| Feature | Type | Meaning | Allowed min | Allowed max | Check | Known issue |
|---|---|---|---|---|---|---|
| `IDpol` | ID | Policy ID (repeats for multi-claim policies) | n/a | n/a | Must exist in `freq` (checked in IV.d) | ⚠️ 195 claims with no matching policy |
| `ClaimAmount` | Numeric | Cost of one claim in euros | > 0 | No limit | Must be positive | 🔍 48 claims under €10 |

**Grouped severity table (`sev_grouped`): one row per policy with at least one claim**

| Feature | Type | Meaning | Allowed min | Allowed max | Check | Known issue |
|---|---|---|---|---|---|---|
| `IDpol` | ID | Policy ID | n/a | n/a | Must be unique (one row per policy) | ✅ |
| `n_claims` | Count | Number of claims on the policy | 1 | No limit | Must be at least 1 (only policies with claims) | ✅ |
| `total_claim_amount` | Numeric | Sum of the policy's claim amounts, in euros | > 0 | No limit | Must be positive | ✅ |

**Table-level checks** (nothing should be lost when grouping)

| Check | Expected | Known issue |
|---|---|---|
| Number of rows = distinct `IDpol` in `sev` | 24,950 | ✅ |
| Sum of `n_claims` = number of rows in `sev` | 26,639 | ✅ |
| Sum of `total_claim_amount` = sum of `ClaimAmount` in `sev` | Equal | ✅ |


In [ ]:
# Define categorical and numerical columns for the frequency dataset
categorical_columns_freq = freq.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_columns_freq = freq.select_dtypes(include=["int64", "float64"]).columns
# Numeric features only (IDpol is an identifier, not a feature): use this for plots
numerical_features_freq = [c for c in numerical_columns_freq if c != "IDpol"]

# Print both lists
print("Categorical columns in the frequency dataset:")
print(categorical_columns_freq)

print("\nNumerical columns in the frequency dataset:")
print(numerical_columns_freq)

In [ ]:
# Display summary statistics for the frequency dataset
# 1. Numerical columns
print("Summary statistics for the frequency dataset:")
freq[numerical_columns_freq].describe().T.round(2)

In [ ]:
# Display summary statistics for the frequency dataset
# 2. Categorical columns
print("Summary statistics for the frequency dataset:")
freq[categorical_columns_freq].describe(include="all").T.round(2)

In [ ]:
# Display summary statistics for the severity dataset
print("Summary statistics for the severity dataset:")
sev.describe(include="all").T

In [ ]:
# Display summary statistics for the grouped severity dataset
print("Summary statistics for the severity dataset:")
sev_grouped.describe(include="all").T

In [ ]:
# Check values range and cardinality in frequency dataset
# Checking that ClaimNb is non-negative (greater than or equal to 0)
if (freq["ClaimNb"] < 0).any():
    print("⚠️ Warning: There are negative values in the 'ClaimNb' column of the frequency dataset.")
else:
    print("All values in the 'ClaimNb' column of the frequency dataset are non-negative.")

# Checking that Exposure is between (0,1]
if (freq["Exposure"] <= 0).any() or (freq["Exposure"] > 1).any():
    print(
        "⚠️ Warning: There are values in the 'Exposure' column of the frequency dataset "
        "that are not between (0,1]."
    )
else:
    print("All values in the 'Exposure' column of the frequency dataset are between (0,1].")

# Checking that Area has valid values: A,B,C,D,E or F
valid_areas = {"A", "B", "C", "D", "E", "F"}
if not freq["Area"].isin(valid_areas).all():
    print(
        "⚠️ Warning: There are values in the 'Area' column of the frequency dataset "
        "that are not in {A, B, C, D, E, F}."
    )
else:
    print("All values in the 'Area' column of the frequency dataset are valid (A to F).")

# Checking that VehPower is greater than 0
if (freq["VehPower"] <= 0).any():
    print(
        "⚠️ Warning: There are values in the 'VehPower' column of the frequency dataset "
        "that are not greater than 0."
    )
else:
    print("All values in the 'VehPower' column of the frequency dataset are greater than 0.")

# Checking that VehAge is non-negative (greater than or equal to 0)
if (freq["VehAge"] < 0).any():
    print("⚠️ Warning: There are negative values in the 'VehAge' column of the frequency dataset.")
else:
    print("All values in the 'VehAge' column of the frequency dataset are non-negative.")

# Checking that DrivAge is greater than or equal to 18
if (freq["DrivAge"] < 18).any():
    print(
        "⚠️ Warning: There are values in the 'DrivAge' column of the frequency dataset "
        "that are less than 18."
    )
else:
    print("All values in the 'DrivAge' column of the frequency dataset are at least 18.")

# Checking that BonusMalus is between 50 and 350
if (freq["BonusMalus"] < 50).any() or (freq["BonusMalus"] > 350).any():
    print(
        "⚠️ Warning: There are values in the 'BonusMalus' column of the frequency dataset "
        "outside [50, 350]."
    )
else:
    print("All values in the 'BonusMalus' column of the frequency dataset are within [50, 350].")

# Checking that VehBrand has valid brand codes
valid_brands = {"B1", "B2", "B3", "B4", "B5", "B6", "B10", "B11", "B12", "B13", "B14"}
if not freq["VehBrand"].isin(valid_brands).all():
    print(
        "⚠️ Warning: There are unknown brand codes in the 'VehBrand' column of the "
        "frequency dataset."
    )
else:
    print("All values in the 'VehBrand' column of the frequency dataset are known brand codes.")

# Checking that VehGas is either Diesel or Regular
valid_gas = {"Diesel", "Regular"}
if not freq["VehGas"].isin(valid_gas).all():
    print(
        "⚠️ Warning: There are values in the 'VehGas' column of the frequency dataset "
        "that are not Diesel or Regular."
    )
else:
    print("All values in the 'VehGas' column of the frequency dataset are Diesel or Regular.")

# Checking that Density is greater than 0
if (freq["Density"] <= 0).any():
    print(
        "⚠️ Warning: There are values in the 'Density' column of the frequency dataset "
        "that are not greater than 0."
    )
else:
    print("All values in the 'Density' column of the frequency dataset are greater than 0.")

# Checking that Region has valid region codes
valid_regions = {
    "R11",
    "R21",
    "R22",
    "R23",
    "R24",
    "R25",
    "R26",
    "R31",
    "R41",
    "R42",
    "R43",
    "R52",
    "R53",
    "R54",
    "R72",
    "R73",
    "R74",
    "R82",
    "R83",
    "R91",
    "R93",
    "R94",
}
if not freq["Region"].isin(valid_regions).all():
    print(
        "⚠️ Warning: There are unknown region codes in the 'Region' column of the frequency dataset."
    )
else:
    print("All values in the 'Region' column of the frequency dataset are known region codes.")

# Checking that IDpol is unique in the frequency dataset
if freq["IDpol"].duplicated().any():
    print("⚠️ Warning: There are duplicate values in the 'IDpol' column of the frequency dataset.")
else:
    print("All values in the 'IDpol' column of the frequency dataset are unique.")

# Checking that ClaimAmount is greater than 0 (severity dataset)
if (sev["ClaimAmount"] <= 0).any():
    print(
        "⚠️ Warning: There are values in the 'ClaimAmount' column of the severity dataset "
        "that are not greater than 0."
    )
else:
    print("All values in the 'ClaimAmount' column of the severity dataset are greater than 0.")

In [ ]:
# Check values range and cardinality in severity dataset
# Checks on the grouped severity table
# Checking that IDpol is unique
if sev_grouped["IDpol"].duplicated().any():
    print("⚠️ Warning: There are duplicate values in the 'IDpol' column of the grouped table.")
else:
    print("All values in the 'IDpol' column of the grouped table are unique.")

# Checking that n_claims is at least 1
if (sev_grouped["n_claims"] < 1).any():
    print("⚠️ Warning: There are values in the 'n_claims' column of the grouped table below 1.")
else:
    print("All values in the 'n_claims' column of the grouped table are at least 1.")

# Checking that total_claim_amount is greater than 0
if (sev_grouped["total_claim_amount"] <= 0).any():
    print(
        "⚠️ Warning: There are values in the 'total_claim_amount' column of the grouped "
        "table that are not greater than 0."
    )
else:
    print("All values in the 'total_claim_amount' column of the grouped table are positive.")

# Checking that no claims were lost or double-counted when grouping
if len(sev_grouped) != sev["IDpol"].nunique():
    print("⚠️ Warning: Grouped row count does not match the number of distinct policies in sev.")
else:
    print(f"Grouped table has one row per policy ({len(sev_grouped):,} rows).")

if sev_grouped["n_claims"].sum() != len(sev):
    print("⚠️ Warning: Total n_claims does not match the number of claims in sev.")
else:
    print(f"Total n_claims matches the number of claims in sev ({len(sev):,}).")

if not np.isclose(sev_grouped["total_claim_amount"].sum(), sev["ClaimAmount"].sum()):
    print("⚠️ Warning: Total claim amount does not match the sum of ClaimAmount in sev.")
else:
    print("Total claim amount matches the sum of ClaimAmount in sev.")

In [ ]:
# Count the number of rows with exposure greater than 1 in the frequency dataset
exposure_greater_than_one_count = (freq["Exposure"] > 1).sum()
print(f"Number of rows in 'Exposure' with value greater than 1: {exposure_greater_than_one_count}")

# Select the rows where Exposure is greater than 1
exposure_greater_than_one_rows = freq[freq["Exposure"] > 1]

# Display a sample of the rows with exposure greater than 1
exposure_greater_than_one_rows.sample(100)

In [ ]:
# Label each policy with its IDpol block (IDs come in separate blocks of ~1M: 0M, 1M, ..., 6M)
# Data-quality label only: used to check whether issues cluster by batch, NOT a model feature
freq["id_block"] = pd.cut(
    freq["IDpol"],
    bins=[0, 1e6, 2e6, 3e6, 4e6, 5e6, 6e6, 7e6],
    labels=[f"{i}M" for i in range(7)],
)
freq["id_block"].value_counts().sort_index()

In [ ]:
# Plot the distribution of each feature for policies with exposure greater than 1
over = freq["Exposure"] > 1
groups = {"Exposure ≤ 1": (freq[~over], "#0836cc"), "Exposure > 1": (freq[over], "#d1242f")}

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
panel_cols = numerical_features_freq + categorical_columns_freq + ["id_block"]
for ax, col in zip(axes.flat, panel_cols, strict=True):
    if col in numerical_features_freq:
        bins = np.histogram_bin_edges(freq[col], bins=40)  # same bins for both groups
        for label, (df, color) in groups.items():
            ax.hist(
                df[col],
                bins=bins,
                density=True,
                histtype="step",
                linewidth=2,
                color=color,
                label=label,
            )
    else:
        shares = pd.DataFrame(
            {label: df[col].value_counts(normalize=True) for label, (df, _) in groups.items()}
        )
        x = np.arange(len(shares))
        for i, (label, (_, color)) in enumerate(groups.items()):
            ax.bar(x + (i - 0.5) * 0.4, shares[label], width=0.4, color=color, label=label)
        ax.set_xticks(x, shares.index.astype(str), rotation=90, fontsize=8)
    ax.set_title(col)
    ax.spines[["top", "right"]].set_visible(False)

axes.flat[0].legend(frameon=False)
fig.suptitle("Exposure > 1 rows vs all other policies (share of each group)", fontsize=14)
fig.tight_layout()
plt.show()

**🔍 Observations: Exposure > 1 vs all other policies** *(to investigate later)*

- The exposure > 1 rows might come from specific regional batches (they also sit at the end of the 1M and 2M IDpol blocks), possibly with a different exposure definition.

**Impact on this project:** none for now. Exposure is not a severity feature.


In [ ]:
# Count the number of rows with vehAge greater than 30 in the frequency dataset
veh_age_greater_than_30_count = (freq["VehAge"] > 30).sum()
print(
    "Number of rows in the 'VehAge' column of the frequency dataset "
    f"that are greater than 30: {veh_age_greater_than_30_count}"
)

# Select the rows where vehAge is greater than 30
veh_age_greater_than_30_rows = freq[freq["VehAge"] > 30]

# Display a sample of the rows with vehAge greater than 30
veh_age_greater_than_30_rows.sample(10)

In [ ]:
# Plot the distribution of each feature for policies with vehAge greater than 30
df = veh_age_greater_than_30_rows

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
panel_cols = numerical_features_freq + categorical_columns_freq + ["id_block"]
for ax, col in zip(axes.flat, panel_cols, strict=True):
    if col in numerical_features_freq:
        ax.hist(
            df[col], bins=40, weights=np.ones(len(df)) / len(df), color="#c57e22", edgecolor="white"
        )
    else:
        shares = df[col].value_counts(normalize=True)
        ax.bar(shares.index.astype(str), shares.values, color="#c57e22")
        ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.set_title(col)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Distribution of each feature: old vehicles ({len(df):,} policies)", fontsize=14)
fig.tight_layout()
plt.show()

### IV.d Join integrity (freq ↔ sev)